# PDF Generator - FAANG Master Guide

**Objetivo:** Gerar PDF limpo e interativo com paleta de cores clean (light backgrounds + dark code)

## Instalações Necessárias

In [ ]:
# Instalar dependências (rode uma vez)
import subprocess
import sys

packages = ['fpdf2', 'pygments']
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} já instalado")
    except ImportError:
        print(f"Instalando {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f"✓ {package} instalado")

## Carregar & Processar Markdown

In [ ]:
import os
import re
from pathlib import Path

# Ler markdown
md_file = 'FAANG_MASTER_GUIDE.md'
pdf_file = 'FAANG_MASTER_GUIDE.pdf'
html_file = 'FAANG_MASTER_GUIDE.html'

print(f"📖 Lendo {md_file}...")
with open(md_file, 'r', encoding='utf-8') as f:
    md_content = f.read()

print(f"✓ Carregado: {len(md_content)} caracteres")
print(f"✓ Linhas: {len(md_content.split(chr(10)))}")

## Gerar HTML Interativo com Cores Limpas

In [ ]:
def markdown_to_clean_html(md_content):
    """Converte markdown para HTML com paleta light + dark code"""
    
    # Paleta de cores CLEAN
    colors = {
        'primary': '#2563eb',      # Azul vibrante
        'secondary': '#1e40af',    # Azul escuro
        'accent': '#059669',       # Verde
        'light_bg': '#f8f9fa',     # Cinza claro
        'border': '#e5e7eb',       # Cinza borda
        'text': '#1f2937',         # Cinza escuro (texto)
        'code_bg': '#1e293b',      # Preto/cinza escuro (código)
        'code_text': '#e2e8f0',    # Branco/cinza claro (texto código)
    }
    
    css = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>FAANG Master Guide - Pattern Recognition</title>
        <style>
            * {{ margin: 0; padding: 0; box-sizing: border-box; }}
            
            body {{
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                line-height: 1.7;
                color: {colors['text']};
                background: white;
                padding: 40px 20px;
            }}
            
            .container {{
                max-width: 950px;
                margin: 0 auto;
            }}
            
            h1 {{
                font-size: 2.5em;
                color: white;
                background: linear-gradient(135deg, {colors['primary']}, {colors['secondary']});
                padding: 50px 40px;
                margin: -40px -20px 40px -20px;
                border-radius: 0;
                text-shadow: 0 2px 4px rgba(0,0,0,0.1);
            }}
            
            h2 {{
                font-size: 1.8em;
                color: {colors['primary']};
                margin: 50px 0 20px;
                border-bottom: 3px solid {colors['primary']};
                padding-bottom: 15px;
                page-break-after: avoid;
            }}
            
            h3 {{
                font-size: 1.4em;
                color: {colors['secondary']};
                margin: 30px 0 15px;
                page-break-after: avoid;
            }}
            
            h4 {{
                font-size: 1.1em;
                color: {colors['text']};
                margin: 20px 0 10px;
            }}
            
            p {{
                margin: 15px 0;
                text-align: justify;
                font-size: 15px;
            }}
            
            code {{
                background-color: {colors['light_bg']};
                color: {colors['secondary']};
                padding: 3px 8px;
                border-radius: 4px;
                font-family: 'Courier New', monospace;
                font-size: 13px;
                font-weight: 500;
            }}
            
            pre {{
                background-color: {colors['code_bg']};
                color: {colors['code_text']};
                padding: 20px;
                border-radius: 8px;
                overflow-x: auto;
                font-family: 'Courier New', monospace;
                font-size: 12px;
                line-height: 1.6;
                margin: 25px 0;
                border-left: 4px solid {colors['accent']};
                page-break-inside: avoid;
                box-shadow: 0 4px 6px rgba(0,0,0,0.1);
            }}
            
            pre code {{
                background: none;
                color: inherit;
                padding: 0;
                border-radius: 0;
                font-size: 12px;
            }}
            
            table {{
                border-collapse: collapse;
                width: 100%;
                margin: 25px 0;
                page-break-inside: avoid;
                box-shadow: 0 2px 4px rgba(0,0,0,0.05);
                border-radius: 8px;
                overflow: hidden;
            }}
            
            table th, table td {{
                border: 1px solid {colors['border']};
                padding: 14px;
                text-align: left;
                font-size: 14px;
            }}
            
            table th {{
                background-color: {colors['light_bg']};
                color: {colors['text']};
                font-weight: 600;
                border-bottom: 2px solid {colors['primary']};
            }}
            
            table tr:hover {{
                background-color: {colors['light_bg']};
            }}
            
            ul, ol {{
                margin: 15px 0 15px 30px;
            }}
            
            li {{
                margin: 8px 0;
                font-size: 15px;
            }}
            
            blockquote {{
                border-left: 4px solid {colors['accent']};
                padding: 15px 20px;
                margin: 20px 0;
                background-color: {colors['light_bg']};
                border-radius: 4px;
                color: {colors['text']};
                font-style: italic;
            }}
            
            hr {{
                margin: 40px 0;
                border: none;
                border-top: 2px solid {colors['border']};
            }}
            
            @media print {{
                body {{ background: white; padding: 0; }}
                h1 {{ margin: 0 0 30px 0; page-break-after: avoid; }}
                pre {{ page-break-inside: avoid; }}
                table {{ page-break-inside: avoid; }}
            }}
        </style>
    </head>
    <body>
        <div class="container">
    """
    
    # Parse markdown
    lines = md_content.split('\n')
    html_content = []
    in_code_block = False
    code_buffer = []
    code_lang = 'python'
    
    i = 0
    while i < len(lines):
        line = lines[i]
        
        # Code blocks
        if line.startswith('```'):
            if not in_code_block:
                in_code_block = True
                code_lang = line.replace('```', '').strip() or 'python'
                code_buffer = []
            else:
                in_code_block = False
                code_text = '\n'.join(code_buffer)
                # Escape HTML
                code_text = code_text.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
                html_content.append(f'<pre><code>{code_text}</code></pre>')
                code_buffer = []
            i += 1
            continue
        
        if in_code_block:
            code_buffer.append(line)
            i += 1
            continue
        
        # Títulos
        if line.startswith('# '):
            text = line.replace('# ', '', 1).strip()
            html_content.append(f'<h1>{text}</h1>')
        elif line.startswith('## '):
            text = line.replace('## ', '', 1).strip()
            html_content.append(f'<h2>{text}</h2>')
        elif line.startswith('### '):
            text = line.replace('### ', '', 1).strip()
            html_content.append(f'<h3>{text}</h3>')
        elif line.startswith('#### '):
            text = line.replace('#### ', '', 1).strip()
            html_content.append(f'<h4>{text}</h4>')
        elif line.strip() in ['---', '***', '___']:
            html_content.append('<hr>')
        elif line.strip().startswith('- '):
            text = line.strip()[2:]
            text = apply_inline_formatting(text)
            html_content.append(f'<li>{text}</li>')
        elif line.strip() and not line.startswith('|'):
            text = line.strip()
            if text:
                text = apply_inline_formatting(text)
                html_content.append(f'<p>{text}</p>')
        
        i += 1
    
    html = css + '\n'.join(html_content) + '\n        </div>\n    </body>\n</html>'
    return html

def apply_inline_formatting(text):
    """Aplica formatação inline (bold, italic, code)"""
    text = re.sub(r'`([^`]+)`', r'<code>\1</code>', text)
    text = re.sub(r'\*\*([^*]+)\*\*', r'<strong>\1</strong>', text)
    text = re.sub(r'\*([^*]+)\*', r'<em>\1</em>', text)
    return text

print("✓ Funções de conversão carregadas")

## Gerar HTML

In [ ]:
print("🔄 Convertendo Markdown → HTML...")
html = markdown_to_clean_html(md_content)

print(f"💾 Salvando {html_file}...")
with open(html_file, 'w', encoding='utf-8') as f:
    f.write(html)

file_size = os.path.getsize(html_file) / 1024
print(f"✅ HTML gerado: {html_file}")
print(f"📊 Tamanho: {file_size:.1f} KB")
print(f"\n💡 Dica: Abra no navegador e use Ctrl+P para imprimir como PDF")

## Gerar PDF com FPDF2

In [ ]:
from fpdf import FPDF

print("📝 Gerando PDF com FPDF2...")

pdf = FPDF(format='A4')
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.set_font("Helvetica", "", 10)

lines = md_content.split('\n')
page_count = 0

for line in lines:
    try:
        # Remove emojis
        clean_line = line.encode('ascii', 'ignore').decode('ascii')
        
        if clean_line.startswith('# '):
            pdf.set_font("Helvetica", "B", 16)
            text = clean_line.replace('# ', '').strip()
            if text:
                pdf.cell(0, 10, text, new_x="LMARGIN", new_y="NEXT")
            pdf.ln(3)
            pdf.set_font("Helvetica", "", 10)
            
        elif clean_line.startswith('## '):
            pdf.set_font("Helvetica", "B", 13)
            text = clean_line.replace('## ', '').strip()
            if text:
                pdf.cell(0, 10, text, new_x="LMARGIN", new_y="NEXT")
            pdf.ln(2)
            pdf.set_font("Helvetica", "", 10)
            
        elif clean_line.startswith('### '):
            pdf.set_font("Helvetica", "B", 11)
            text = clean_line.replace('### ', '').strip()
            if text:
                pdf.cell(0, 10, text, new_x="LMARGIN", new_y="NEXT")
            pdf.ln(1)
            pdf.set_font("Helvetica", "", 10)
            
        elif clean_line.strip() and not clean_line.startswith('#') and not clean_line.startswith('```'):
            pdf.set_font("Helvetica", "", 9)
            text = clean_line.strip()
            if len(text) > 85:
                pdf.multi_cell(0, 4, text)
            else:
                if text:
                    pdf.cell(0, 4, text, new_x="LMARGIN", new_y="NEXT")
        
        elif not clean_line.strip():
            pdf.ln(1)
    
    except Exception as e:
        pass  # Skip problematic lines

pdf.output(pdf_file)

file_size = os.path.getsize(pdf_file) / (1024 * 1024)
print(f"✅ PDF gerado: {pdf_file}")
print(f"📊 Tamanho: {file_size:.2f} MB")
print(f"✓ Páginas: {pdf.page}")

## Resumo Final

In [ ]:
print("\n" + "="*60)
print("📚 FAANG Master Guide - Exportação Completa")
print("="*60)
print(f"\n✅ Origem: {md_file}")
print(f"\n📄 Arquivos Gerados:")
print(f"   1. {html_file} - HTML interativo com cores clean")
print(f"      → Abra no navegador para melhor visualização")
print(f"      → Use Ctrl+P para imprimir como PDF")
print(f"\n   2. {pdf_file} - PDF direto (FPDF2)")
print(f"      → Pronto para compartilhar")
print(f"\n💾 Total: {os.path.getsize(html_file)/1024 + os.path.getsize(pdf_file)/(1024*1024):.2f} MB")
print(f"\n🎨 Paleta de cores:")
print(f"   • Backgrounds: Light (#f8f9fa)")
print(f"   • Código: Dark (#1e293b)")
print(f"   • Texto: Cinza escuro (#1f2937)")
print(f"   • Primário: Azul (#2563eb)")
print("\n" + "="*60)